# Tmap API + 회피 알고리즘 재설계

In [1]:
# 셀 1: 라이브러리 import 및 환경 점검
# v2 노트북 시작 — Tmap 도보 polyline 기반 회피 영역 검증

# 표준 라이브러리
import os
import json
import time
import math
import itertools
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

# HTTP·환경
import requests
from dotenv import load_dotenv

# 데이터 처리
import numpy as np
import pandas as pd

# 거리·공간 처리
from shapely.geometry import shape, Point, LineString
from shapely.ops import unary_union
from shapely.prepared import prep
from sklearn.neighbors import BallTree   # 단계 6에서 neighbor_200m 계산용

# 지도 시각화
import folium

print(f"[v2 노트북 시작 — Tmap 도보 polyline 기반]")
print(f"  pandas    {pd.__version__}")
print(f"  numpy     {np.__version__}")
print(f"  folium    {folium.__version__}")
print(f"  requests  {requests.__version__}")
print(f"  작업 디렉토리: {os.getcwd()}")

[v2 노트북 시작 — Tmap 도보 polyline 기반]
  pandas    3.0.3
  numpy     2.4.6
  folium    0.20.0
  requests  2.34.2
  작업 디렉토리: d:\yyeong\ploggo\algorithm-test


In [2]:
# 셀 2 (수정): 인증키 로드 + 출력 디렉토리를 output2로 분리
# v1 산출물(./output)과 v2 산출물(./output2)을 명확히 구분

load_dotenv(override=True)

VWORLD_KEY = os.getenv('VWORLD_KEY')
TMAP_APP_KEY = os.getenv('TMAP_APP_KEY')

if not VWORLD_KEY:
    raise RuntimeError(".env에 VWORLD_KEY가 없다.")
if not TMAP_APP_KEY:
    raise RuntimeError(".env에 TMAP_APP_KEY가 없다.")

print(f"[인증키 로드 완료]")
print(f"  V-World:  {VWORLD_KEY[:4]}...{VWORLD_KEY[-4:]} ({len(VWORLD_KEY)}자)")
print(f"  Tmap:     {TMAP_APP_KEY[:6]}...{TMAP_APP_KEY[-4:]} ({len(TMAP_APP_KEY)}자)")

# 출력 디렉토리 — v2 전용 폴더로 분리 (v1의 ./output과 공존)
OUTPUT_DIR = Path("./output2")
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"  출력 디렉토리: {OUTPUT_DIR.resolve()}")
print(f"  (v1 산출물은 ./output에 그대로 보존)")

[인증키 로드 완료]
  V-World:  8C07...3D4E (36자)
  Tmap:     r8mALt...H5Zm (40자)
  출력 디렉토리: D:\yyeong\ploggo\algorithm-test\output2
  (v1 산출물은 ./output에 그대로 보존)


In [3]:
# 셀 3: v1 회피 영역 GeoJSON 로드 + shapely 변환
# v1 노트북에서 V-World로 수집한 147개 폴리곤을 그대로 활용 (시간 절약)

from shapely.geometry import shape
from shapely.ops import unary_union
from shapely.prepared import prep

# v1 산출물 경로 (output 폴더에 있음)
GEOJSON_PATH = Path("./output/incheon_avoid_zones_v1_260601.geojson")

if not GEOJSON_PATH.exists():
    raise FileNotFoundError(
        f"v1 GeoJSON 파일이 없다: {GEOJSON_PATH.resolve()}\n"
        f"v1 노트북의 셀 11까지 실행한 산출물이 있어야 한다."
    )

with open(GEOJSON_PATH, 'r', encoding='utf-8') as f:
    avoid_geojson = json.load(f)

# 147개 features → shapely 폴리곤 + 메타데이터
avoid_polygons = []
for feature in avoid_geojson['features']:
    props = feature['properties']
    geom = shape(feature['geometry'])
    avoid_polygons.append({
        'geom':       geom,
        'uqa_code':   props.get('uqa_code'),
        'uname':      props.get('uname'),
        'sigg_name':  props.get('sigg_name'),
        'feature_id': props.get('feature_id'),
    })

# 전체 폴리곤 union (교차 검사·페널티 계산용)
avoid_union = unary_union([p['geom'] for p in avoid_polygons])
avoid_union_prepared = prep(avoid_union)  # 점-폴리곤 검사 가속

# UQA 코드별 union (분석용)
avoid_union_by_code = {}
for code in ['UQA310', 'UQA320', 'UQA330']:
    geoms = [p['geom'] for p in avoid_polygons if p['uqa_code'] == code]
    if geoms:
        avoid_union_by_code[code] = unary_union(geoms)

# 통계 출력
print(f"[회피 영역 로드 완료]")
print(f"  총 폴리곤: {len(avoid_polygons)}개")
print(f"  UQA 코드별:")
for code in ['UQA310', 'UQA320', 'UQA330']:
    n = sum(1 for p in avoid_polygons if p['uqa_code'] == code)
    print(f"    {code}: {n}개")

print(f"\n  군구별:")
sigg_counts = Counter(p['sigg_name'] for p in avoid_polygons)
for sigg, n in sigg_counts.most_common():
    print(f"    {sigg:<15} {n}개")

print(f"\n  avoid_union bounds (lon_min, lat_min, lon_max, lat_max):")
print(f"    {avoid_union.bounds}")

[회피 영역 로드 완료]
  총 폴리곤: 147개
  UQA 코드별:
    UQA310: 5개
    UQA320: 46개
    UQA330: 96개

  군구별:
    서구              44개
    중구              29개
    미추홀구            15개
    남동구             14개
    동구              12개
    계양구             12개
    연수구             11개
    부평구             7개
    강화군             3개

  avoid_union bounds (lon_min, lat_min, lon_max, lat_max):
    (126.47063840934426, 37.34180714965152, 126.76623837593196, 37.767857612646296)


In [4]:
# 셀 4: 핫스팟 CSV 로드 + neighbor_200m + S_i 계산
# routeTest_v3 셀 3·5·6 이식 (핸드오프 v4 알고리즘 사양 그대로)

import ast
from sklearn.neighbors import BallTree

# === 1. CSV 로드 ===
CSV_PATH = Path("./hotspots_CCTV_20260510.csv")
if not CSV_PATH.exists():
    raise FileNotFoundError(f"핫스팟 CSV가 없다: {CSV_PATH.resolve()}")

hotspots = pd.read_csv(CSV_PATH, encoding='utf-8-sig')

# list 필드 복원 (문자열 → list)
def parse_list_field(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return None
    return value

hotspots['install_years']      = hotspots['install_years'].apply(parse_list_field)
hotspots['relocation_history'] = hotspots['relocation_history'].apply(parse_list_field)

print(f"[핫스팟 로드] {len(hotspots):,}곳, {len(hotspots.columns)}컬럼")

# === 2. neighbor_200m 계산 (BallTree) ===
# 핸드오프 D2-2 공식: S_i = cctv_count + 0.5 × neighbor_200m
EARTH_RADIUS_KM = 6371.0

t0 = time.time()
coords_rad = np.radians(hotspots[['latitude', 'longitude']].values)
tree = BallTree(coords_rad, metric='haversine')
radius_rad = 0.2 / EARTH_RADIUS_KM   # 200m → 라디안

# query_radius는 자기 자신 포함 카운트 반환 → -1 보정
counts = tree.query_radius(coords_rad, r=radius_rad, count_only=True)
hotspots['neighbor_200m'] = counts - 1

print(f"\n[neighbor_200m 계산] 소요 {time.time() - t0:.2f}초")
print(f"  분포: min {hotspots['neighbor_200m'].min()}, "
      f"max {hotspots['neighbor_200m'].max()}, "
      f"평균 {hotspots['neighbor_200m'].mean():.2f}")

n_with_neighbor = (hotspots['neighbor_200m'] > 0).sum()
print(f"  200m 내 이웃 보유: {n_with_neighbor:,}곳 ({n_with_neighbor/len(hotspots)*100:.1f}%)")

# === 3. S_i 계산 ===
hotspots['S_i'] = hotspots['cctv_count'] + 0.5 * hotspots['neighbor_200m']

print(f"\n[S_i 계산 완료]")
print(f"  분포: min {hotspots['S_i'].min():.2f}, "
      f"max {hotspots['S_i'].max():.2f}, "
      f"평균 {hotspots['S_i'].mean():.2f}")

print(f"\n  군구별 S_i 평균:")
for sigg, mean_si in hotspots.groupby('source_district')['S_i'].mean().items():
    print(f"    {sigg:<12} {mean_si:.2f}")

# === 4. 회피 영역 내부 핫스팟 표시 (방법 C에서 활용) ===
# 각 핫스팟이 회피 영역 내부에 있는지 미리 계산
def is_in_avoid_zone(row):
    pt = Point(row['longitude'], row['latitude'])
    return avoid_union_prepared.contains(pt)

t0 = time.time()
hotspots['in_avoid_zone'] = hotspots.apply(is_in_avoid_zone, axis=1)
print(f"\n[회피 영역 내부 검사] 소요 {time.time() - t0:.2f}초")
print(f"  회피 영역 내부 핫스팟: {hotspots['in_avoid_zone'].sum()}/{len(hotspots):,} "
      f"({hotspots['in_avoid_zone'].sum()/len(hotspots)*100:.2f}%)")

[핫스팟 로드] 1,535곳, 15컬럼

[neighbor_200m 계산] 소요 0.01초
  분포: min 0, max 17, 평균 3.16
  200m 내 이웃 보유: 1,178곳 (76.7%)

[S_i 계산 완료]
  분포: min 1.00, max 11.00, 평균 2.78

  군구별 S_i 평균:
    bupyeong     1.25
    donggu       1.88
    ganghwa      1.07
    gyeyang      3.51
    junggu       1.94
    michuhol     2.11
    namdong      1.70
    seogu        2.65
    yeonsu       4.86

[회피 영역 내부 검사] 소요 0.03초
  회피 영역 내부 핫스팟: 58/1,535 (3.78%)


In [5]:
# 셀 5: 거리·Buffer 유틸 함수 + OD 9쌍 + 후보 추출 검증
# haversine 패키지 의존 제거 — numpy 기반 직접 구현 (수학적으로 동일)

# === 0. Haversine 직접 구현 ===
def haversine_km(p_a, p_b):
    """두 (lat, lon) 좌표 사이의 Haversine 거리 (km)
    
    haversine 패키지의 hs.haversine(p_a, p_b, unit=hs.Unit.KILOMETERS)와 동일 결과.
    지구 반경 6371.0 km 사용.
    """
    lat1, lon1 = p_a
    lat2, lon2 = p_b
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    return EARTH_RADIUS_KM * c   # EARTH_RADIUS_KM은 셀 4에서 6371.0으로 정의됨

# === 1. 도보·Buffer 유틸 함수 ===
DETOUR_RATIO = 1.3   # 한국 도시 도보 환경 평균 보정계수 (핸드오프 D3)

def walking_distance_km(p_a, p_b):
    """추정 도보 거리 (km) = Haversine × 1.3"""
    return haversine_km(p_a, p_b) * DETOUR_RATIO

def detour_distance_km(origin, hotspot, destination):
    """핫스팟 단독 경유 시 우회 거리: D_i = d(O,h) + d(h,D) - d(O,D)"""
    d_oh = walking_distance_km(origin, hotspot)
    d_hd = walking_distance_km(hotspot, destination)
    d_od = walking_distance_km(origin, destination)
    return max(d_oh + d_hd - d_od, 0.0)

def get_buffer_radius(district):
    """군구별 buffer 반경 (km) — 강화군 2.5km, 그 외 1.5km (핸드오프 D4)"""
    return 2.5 if district == 'ganghwa' else 1.5

def filter_candidates_by_ellipse_buffer(df, origin, destination, buffer_r_km):
    """타원 buffer 필터링: d(O,h) + d(h,D) - d(O,D) < buffer_r × 1.5"""
    d_od = walking_distance_km(origin, destination)
    threshold = buffer_r_km * 1.5
    
    coords = df[['latitude', 'longitude']].values
    d_oh = np.array([walking_distance_km(origin, tuple(c)) for c in coords])
    d_hd = np.array([walking_distance_km(tuple(c), destination) for c in coords])
    
    mask = (d_oh + d_hd - d_od) < threshold
    return df[mask].copy()

# === 2. OD 9쌍 좌표 (routeTest_v3 OD_CANDIDATES_V2와 동일) ===
OD_CANDIDATES_V2 = {
    'ganghwa':  {'O': (37.706505, 126.443398), 'D': (37.732838, 126.517697)},
    'junggu':   {'O': (37.473565, 126.621778), 'D': (37.474748, 126.603258)},
    'donggu':   {'O': (37.470733, 126.641914), 'D': (37.487780, 126.621754)},
    'michuhol': {'O': (37.455015, 126.667435), 'D': (37.435062, 126.690721)},
    'seogu':    {'O': (37.545115, 126.676062), 'D': (37.552748, 126.650239)},
    'yeonsu':   {'O': (37.391182, 126.645092), 'D': (37.418275, 126.661122)},
    'namdong':  {'O': (37.447245, 126.731453), 'D': (37.447769, 126.700089)},
    'bupyeong': {'O': (37.507046, 126.721858), 'D': (37.482179, 126.703038)},
    'gyeyang':  {'O': (37.537367, 126.737744), 'D': (37.530279, 126.722516)},
}

# === 3. 각 OD에 대해 Buffer 후보 추출 + 표본 편향 검증 ===
print(f"[Buffer 후보 추출 검증]")
print(f"{'군구':<10} {'직선(km)':>9} {'buffer(km)':>11} {'후보수':>7} {'회피_내부':>9}")
print("-" * 55)

od_buffer_results = {}
for district, od in OD_CANDIDATES_V2.items():
    origin = od['O']
    destination = od['D']
    
    direct_km = haversine_km(origin, destination)
    buffer_r = get_buffer_radius(district)
    
    district_df = hotspots[hotspots['source_district'] == district]
    cands = filter_candidates_by_ellipse_buffer(district_df, origin, destination, buffer_r)
    n_avoid_in_buffer = int(cands['in_avoid_zone'].sum())
    
    od_buffer_results[district] = {
        'origin': origin,
        'destination': destination,
        'direct_km': direct_km,
        'buffer_r_km': buffer_r,
        'candidates': cands,
        'n_candidates': len(cands),
        'n_avoid_in_buffer': n_avoid_in_buffer,
    }
    
    print(f"{district:<10} {direct_km:>7.2f}   {buffer_r:>9}   {len(cands):>5}   "
          f"{n_avoid_in_buffer:>6}")

print(f"\n[중요 — 표본 편향 가설 검증]")
total_avoid_in_buffer = sum(r['n_avoid_in_buffer'] for r in od_buffer_results.values())
print(f"  9 OD의 Buffer 안 회피 핫스팟 총합: {total_avoid_in_buffer}곳")
if total_avoid_in_buffer == 0:
    print(f"  → 표본 편향 가설 정량 입증: Buffer 안에 회피 핫스팟이 0건이므로")
    print(f"    옵션 X 0건은 알고리즘 견고성이 아닌 OD 선정 편향의 결과")
elif total_avoid_in_buffer < 5:
    print(f"  → 표본 편향 가설 부분 입증: Buffer 안 회피 핫스팟이 매우 적음")
else:
    print(f"  → 알고리즘이 일부 회피 효과 보일 가능성 (S_i 정렬에서 탈락 등)")

[Buffer 후보 추출 검증]
군구            직선(km)  buffer(km)     후보수     회피_내부
-------------------------------------------------------
ganghwa       7.16         2.5      27        0
junggu        1.64         1.5      28        1
donggu        2.60         1.5      46        7
michuhol      3.02         1.5     130        0
seogu         2.43         1.5      61        0
yeonsu        3.33         1.5      67        0
namdong       2.77         1.5      73        0
bupyeong      3.23         1.5      23        0
gyeyang       1.56         1.5     110        7

[중요 — 표본 편향 가설 검증]
  9 OD의 Buffer 안 회피 핫스팟 총합: 15곳
  → 알고리즘이 일부 회피 효과 보일 가능성 (S_i 정렬에서 탈락 등)


## 정적 회피 페널티 함수 (방법 C 핵심)

In [6]:
# 셀 6: 정적 회피 페널티 함수 (방법 C 핵심)
# Tmap 호출 없이 shapely 직선 교차로 회피 페널티 계산
# 위경도 1도 ≈ 111km (위도 37도 부근) 근사 사용

def compute_avoid_penalty(hotspot_lat, hotspot_lon, origin, destination,
                          avoid_union, avoid_union_prepared):
    """
    핫스팟의 정적 회피 페널티 계산.
    
    구성 요소:
    1. 핫스팟이 회피 영역 내부면 매우 큰 페널티 (999) — 옵션 X 완전 차단
    2. O → 핫스팟 직선의 회피 영역 통과 길이
    3. 핫스팟 → D 직선의 회피 영역 통과 길이
    
    단위: km (다른 항과 일치)
    호출 비용: O(1), 약 1~2 밀리초
    
    Args:
        hotspot_lat, hotspot_lon: 핫스팟 좌표
        origin: (lat, lon) 출발지
        destination: (lat, lon) 도착지
        avoid_union: shapely MultiPolygon (회피 영역 union)
        avoid_union_prepared: prep된 union (점-폴리곤 검사 가속)
    
    Returns:
        float: 페널티 (km)
    """
    # 1. 회피 영역 내부 검사 (옵션 X 차단)
    hotspot_point = Point(hotspot_lon, hotspot_lat)
    if avoid_union_prepared.contains(hotspot_point):
        return 999.0
    
    # 2. O → 핫스팟 직선의 회피 영역 통과
    line_o = LineString([
        (origin[1], origin[0]),
        (hotspot_lon, hotspot_lat),
    ])
    inter_o = line_o.intersection(avoid_union)
    len_o_deg = inter_o.length if not inter_o.is_empty else 0
    
    # 3. 핫스팟 → D 직선의 회피 영역 통과
    line_d = LineString([
        (hotspot_lon, hotspot_lat),
        (destination[1], destination[0]),
    ])
    inter_d = line_d.intersection(avoid_union)
    len_d_deg = inter_d.length if not inter_d.is_empty else 0
    
    # 위경도 → km 근사 (위도 37도 부근 1도 ≈ 111km)
    return (len_o_deg + len_d_deg) * 111.0


# === 동작 검증 — donggu·gyeyang의 회피 후보 vs 일반 후보 페널티 비교 ===
print("[정적 회피 페널티 함수 검증]")
print("=" * 70)

for district in ['donggu', 'gyeyang', 'michuhol']:   # 회피 7곳·7곳·0곳 비교
    od = OD_CANDIDATES_V2[district]
    cands = od_buffer_results[district]['candidates']
    
    # 회피 영역 내부 후보의 페널티
    avoid_cands = cands[cands['in_avoid_zone']]
    
    print(f"\n[{district}] OD: O{od['O']} → D{od['D']}")
    print(f"  Buffer 후보 {len(cands)}곳 중 회피 내부 {len(avoid_cands)}곳")
    
    if len(avoid_cands) > 0:
        print(f"\n  회피 영역 내부 후보 (S_i 상위 3개):")
        for _, row in avoid_cands.nlargest(3, 'S_i').iterrows():
            penalty = compute_avoid_penalty(
                row['latitude'], row['longitude'],
                od['O'], od['D'],
                avoid_union, avoid_union_prepared,
            )
            print(f"    {row['hotspot_id']}: S_i={row['S_i']:.2f}, "
                  f"penalty={penalty:.2f}")
    
    # 비교: 회피 영역 밖 후보 상위 3
    non_avoid = cands[~cands['in_avoid_zone']]
    print(f"\n  회피 영역 밖 후보 (S_i 상위 3개):")
    for _, row in non_avoid.nlargest(3, 'S_i').iterrows():
        penalty = compute_avoid_penalty(
            row['latitude'], row['longitude'],
            od['O'], od['D'],
            avoid_union, avoid_union_prepared,
        )
        print(f"    {row['hotspot_id']}: S_i={row['S_i']:.2f}, "
              f"penalty={penalty:.2f}")

[정적 회피 페널티 함수 검증]

[donggu] OD: O(37.470733, 126.641914) → D(37.48778, 126.621754)
  Buffer 후보 46곳 중 회피 내부 7곳

  회피 영역 내부 후보 (S_i 상위 3개):
    donggu_wydj1rqd_001: S_i=1.50, penalty=999.00
    donggu_wydj32ez_001: S_i=1.50, penalty=999.00
    donggu_wydj3359_001: S_i=1.50, penalty=999.00

  회피 영역 밖 후보 (S_i 상위 3개):
    donggu_wydj1ydk_001: S_i=3.50, penalty=0.79
    donggu_wydj1vg9_001: S_i=3.00, penalty=0.53
    donggu_wydj1x1h_001: S_i=3.00, penalty=0.36

[gyeyang] OD: O(37.537367, 126.737744) → D(37.530279, 126.722516)
  Buffer 후보 110곳 중 회피 내부 7곳

  회피 영역 내부 후보 (S_i 상위 3개):
    gyeyang_wydj7zt6_001: S_i=3.50, penalty=999.00
    gyeyang_wydj7ztf_001: S_i=3.00, penalty=999.00
    gyeyang_wydjkp9v_001: S_i=3.00, penalty=999.00

  회피 영역 밖 후보 (S_i 상위 3개):
    gyeyang_wydje8me_001: S_i=7.00, penalty=0.00
    gyeyang_wydjeep3_001: S_i=7.00, penalty=0.00
    gyeyang_wydje9mr_001: S_i=6.50, penalty=0.00

[michuhol] OD: O(37.455015, 126.667435) → D(37.435062, 126.690721)
  Buffer 후보 130곳 중 회피 내

In [7]:
# 셀 7: Score 공식 통합 — 방법 C 적용
# 새 공식: Score_i = α·S_i - β·D_i - γ·avoid_penalty_i

# === 하이퍼파라미터 ===
ALPHA = 0.5   # S_i 가중치 (핸드오프 D6 그대로)
BETA  = 0.5   # D_i 가중치 (핸드오프 D6 그대로)
GAMMA = 1.0   # avoid_penalty 가중치 (방법 C 신규, 단계 6 후반 튜닝 대상)

# === 회피 페널티 통합 Score 함수 (방법 C) ===
def compute_scores_with_avoidance(df, origin, destination,
                                   avoid_union, avoid_union_prepared,
                                   alpha=ALPHA, beta=BETA, gamma=GAMMA):
    """방법 C 적용 Score 계산.
    Score_i = α·S_i - β·D_i - γ·avoid_penalty_i
    """
    out = df.copy()
    out['D_i'] = out.apply(
        lambda r: detour_distance_km(origin, (r['latitude'], r['longitude']), destination),
        axis=1
    )
    out['avoid_penalty_i'] = out.apply(
        lambda r: compute_avoid_penalty(
            r['latitude'], r['longitude'],
            origin, destination,
            avoid_union, avoid_union_prepared,
        ),
        axis=1
    )
    out['Score_i'] = alpha * out['S_i'] - beta * out['D_i'] - gamma * out['avoid_penalty_i']
    return out

# === 원본 Score 함수 (방법 C 적용 전, 회귀 검증용) ===
def compute_scores_original(df, origin, destination, alpha=ALPHA, beta=BETA):
    """routeTest_v3 원본 Score (회귀 검증용)."""
    out = df.copy()
    out['D_i'] = out.apply(
        lambda r: detour_distance_km(origin, (r['latitude'], r['longitude']), destination),
        axis=1
    )
    out['avoid_penalty_i'] = 0.0   # 없음 (호환성)
    out['Score_i'] = alpha * out['S_i'] - beta * out['D_i']
    return out

print(f"[Score 공식 정의 완료]")
print(f"  원본: Score_i = α·S_i - β·D_i")
print(f"  방법 C: Score_i = α·S_i - β·D_i - γ·avoid_penalty_i")
print(f"  가중치: α={ALPHA}, β={BETA}, γ={GAMMA} (초기값)")
print(f"  γ 튜닝은 단계 6-D에서 진행 (0.5, 1.0, 2.0, 5.0 비교)")

[Score 공식 정의 완료]
  원본: Score_i = α·S_i - β·D_i
  방법 C: Score_i = α·S_i - β·D_i - γ·avoid_penalty_i
  가중치: α=0.5, β=0.5, γ=1.0 (초기값)
  γ 튜닝은 단계 6-D에서 진행 (0.5, 1.0, 2.0, 5.0 비교)


In [8]:
# 셀 8: Greedy 알고리즘 + 방법 C vs 원본 비교
# routeTest_v3의 algorithm_greedy 이식 + 회피 페널티 적용 버전 추가

K = 3
BF_TOP_N = 15

def algorithm_greedy_v2(candidates_df, origin, destination,
                         avoid_union, avoid_union_prepared,
                         alpha=ALPHA, beta=BETA, gamma=GAMMA, k=K):
    """방법 C 적용 Greedy.
    1. S_i 상위 BF_TOP_N=15 풀 (원본과 동일)
    2. compute_scores_with_avoidance로 Score_i 계산 (회피 페널티 포함)
    3. Score_i 상위 K=3 선택
    """
    t0 = time.time()
    n = len(candidates_df)
    if n == 0:
        return pd.DataFrame(), 0
    k_eff = min(k, n)
    
    pool = candidates_df.nlargest(min(BF_TOP_N, n), 'S_i')
    scored = compute_scores_with_avoidance(
        pool, origin, destination, avoid_union, avoid_union_prepared,
        alpha=alpha, beta=beta, gamma=gamma,
    )
    selected = scored.nlargest(k_eff, 'Score_i')
    return selected, (time.time() - t0) * 1000

def algorithm_greedy_original(candidates_df, origin, destination,
                               alpha=ALPHA, beta=BETA, k=K):
    """routeTest_v3 원본 Greedy (회귀 검증용)."""
    t0 = time.time()
    n = len(candidates_df)
    if n == 0:
        return pd.DataFrame(), 0
    k_eff = min(k, n)
    
    pool = candidates_df.nlargest(min(BF_TOP_N, n), 'S_i')
    scored = compute_scores_original(pool, origin, destination, alpha=alpha, beta=beta)
    selected = scored.nlargest(k_eff, 'Score_i')
    return selected, (time.time() - t0) * 1000


# === 9 OD에 대해 원본 vs 방법 C 직접 비교 ===
print("=" * 80)
print("[Greedy 원본 vs 방법 C 비교 — 9 OD]")
print("=" * 80)

comparison_results = []
for district, od in OD_CANDIDATES_V2.items():
    cands = od_buffer_results[district]['candidates']
    
    # 원본
    sel_orig, _ = algorithm_greedy_original(cands, od['O'], od['D'])
    orig_ids = set(sel_orig['hotspot_id'])
    orig_avoid_count = int(sel_orig['in_avoid_zone'].sum()) if 'in_avoid_zone' in sel_orig.columns else 0
    
    # 방법 C
    sel_v2, _ = algorithm_greedy_v2(cands, od['O'], od['D'], avoid_union, avoid_union_prepared)
    v2_ids = set(sel_v2['hotspot_id'])
    v2_avoid_count = int(sel_v2['in_avoid_zone'].sum()) if 'in_avoid_zone' in sel_v2.columns else 0
    
    # K=3 일치 여부
    if orig_ids == v2_ids:
        diff_status = "동일"
    else:
        diff_status = f"다름 (원본만: {len(orig_ids - v2_ids)}, v2만: {len(v2_ids - orig_ids)})"
    
    # 방법 C의 페널티 합계
    v2_total_penalty = sel_v2['avoid_penalty_i'].sum()
    
    comparison_results.append({
        'district': district,
        'orig_avoid_in_K3': orig_avoid_count,
        'v2_avoid_in_K3': v2_avoid_count,
        'diff_status': diff_status,
        'v2_total_penalty': v2_total_penalty,
    })
    
    print(f"\n[{district}]")
    print(f"  원본 K=3 회피 영역 내부: {orig_avoid_count}개")
    print(f"  방법 C K=3 회피 영역 내부: {v2_avoid_count}개")
    print(f"  K=3 비교: {diff_status}")
    print(f"  방법 C 총 페널티: {v2_total_penalty:.2f}")

# 종합
print(f"\n{'=' * 80}")
print(f"[종합]")
df_compare = pd.DataFrame(comparison_results)
n_diff = sum(1 for r in comparison_results if "다름" in r['diff_status'])
print(f"  K=3 선정이 다른 OD: {n_diff}/9")
print(f"  원본 K=3 회피 내부 총: {df_compare['orig_avoid_in_K3'].sum()}")
print(f"  방법 C K=3 회피 내부 총: {df_compare['v2_avoid_in_K3'].sum()} (페널티 999로 차단)")

[Greedy 원본 vs 방법 C 비교 — 9 OD]

[ganghwa]
  원본 K=3 회피 영역 내부: 0개
  방법 C K=3 회피 영역 내부: 0개
  K=3 비교: 동일
  방법 C 총 페널티: 0.00

[junggu]
  원본 K=3 회피 영역 내부: 0개
  방법 C K=3 회피 영역 내부: 0개
  K=3 비교: 다름 (원본만: 1, v2만: 1)
  방법 C 총 페널티: 0.92

[donggu]
  원본 K=3 회피 영역 내부: 0개
  방법 C K=3 회피 영역 내부: 0개
  K=3 비교: 다름 (원본만: 2, v2만: 2)
  방법 C 총 페널티: 1.22

[michuhol]
  원본 K=3 회피 영역 내부: 0개
  방법 C K=3 회피 영역 내부: 0개
  K=3 비교: 동일
  방법 C 총 페널티: 0.00

[seogu]
  원본 K=3 회피 영역 내부: 0개
  방법 C K=3 회피 영역 내부: 0개
  K=3 비교: 동일
  방법 C 총 페널티: 0.00

[yeonsu]
  원본 K=3 회피 영역 내부: 0개
  방법 C K=3 회피 영역 내부: 0개
  K=3 비교: 동일
  방법 C 총 페널티: 0.00

[namdong]
  원본 K=3 회피 영역 내부: 0개
  방법 C K=3 회피 영역 내부: 0개
  K=3 비교: 동일
  방법 C 총 페널티: 0.00

[bupyeong]
  원본 K=3 회피 영역 내부: 0개
  방법 C K=3 회피 영역 내부: 0개
  K=3 비교: 동일
  방법 C 총 페널티: 0.00

[gyeyang]
  원본 K=3 회피 영역 내부: 0개
  방법 C K=3 회피 영역 내부: 0개
  K=3 비교: 동일
  방법 C 총 페널티: 0.00

[종합]
  K=3 선정이 다른 OD: 2/9
  원본 K=3 회피 내부 총: 0
  방법 C K=3 회피 내부 총: 0 (페널티 999로 차단)


### Tmap 보행자 호출 + polyline 교차 검사

In [9]:
# 셀 9: Tmap 보행자 호출 함수 + polyline 교차 검사
# v1 노트북의 셀 21을 v2에 이식 + 교차 검사 함수 추가

TMAP_PEDESTRIAN_URL = "https://apis.openapi.sk.com/tmap/routes/pedestrian?version=1"

def get_pedestrian_route(origin, destination, waypoints=None, debug=False):
    """Tmap 보행자 경로안내 API 호출.
    
    Args:
        origin, destination: (lat, lon)
        waypoints: [(lat, lon), ...] 또는 None (최대 5개)
    Returns:
        dict: path([(lat, lon), ...]), distance_m, duration_ms, status
    """
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
        "appKey": TMAP_APP_KEY,
    }
    payload = {
        "startX": origin[1], "startY": origin[0],
        "endX": destination[1], "endY": destination[0],
        "startName": "출발지", "endName": "도착지",
        "reqCoordType": "WGS84GEO", "resCoordType": "WGS84GEO",
        "searchOption": "0",
    }
    if waypoints:
        if len(waypoints) > 5:
            return {'path': None, 'distance_m': 0, 'duration_ms': 0,
                    'status': f'ERROR: 경유지 {len(waypoints)}개. Tmap 한도 5개 초과'}
        payload["passList"] = "_".join(f"{wp[1]},{wp[0]}" for wp in waypoints)
    
    try:
        r = requests.post(TMAP_PEDESTRIAN_URL, headers=headers, json=payload, timeout=15)
    except requests.exceptions.RequestException as e:
        return {'path': None, 'distance_m': 0, 'duration_ms': 0,
                'status': f'ERROR: {type(e).__name__}'}
    
    if r.status_code != 200:
        return {'path': None, 'distance_m': 0, 'duration_ms': 0,
                'status': f'ERROR: HTTP {r.status_code}, body: {r.text[:150]}'}
    
    try:
        result = r.json()
    except json.JSONDecodeError:
        return {'path': None, 'distance_m': 0, 'duration_ms': 0, 'status': 'JSON_ERROR'}
    
    if result.get('type') != 'FeatureCollection':
        return {'path': None, 'distance_m': 0, 'duration_ms': 0, 'status': 'INVALID'}
    
    features = result.get('features', [])
    if not features:
        return {'path': None, 'distance_m': 0, 'duration_ms': 0, 'status': 'NO_FEATURES'}
    
    first_props = features[0].get('properties', {})
    total_distance_m = first_props.get('totalDistance', 0)
    total_time_s = first_props.get('totalTime', 0)
    
    path_lonlat = []
    for f in features:
        geom = f.get('geometry', {})
        if geom.get('type') == 'LineString':
            path_lonlat.extend(geom.get('coordinates', []))
    
    # (lat, lon) 형식으로 변환 (본 노트북 일관성)
    path_latlon = [(pt[1], pt[0]) for pt in path_lonlat]
    
    return {
        'path': path_latlon,
        'distance_m': total_distance_m,
        'duration_ms': total_time_s * 1000,
        'status': 'OK',
    }


def check_polyline_avoid_intersection(path_latlon, avoid_union_param, 
                                        avoid_union_by_code_param):
    """polyline이 회피 영역과 교차하는지 검사.
    
    Args:
        path_latlon: [(lat, lon), ...] polyline 좌표 리스트
        avoid_union_param: shapely MultiPolygon (전체 회피 영역)
        avoid_union_by_code_param: dict {uqa_code: union} (코드별 회피 영역)
    
    Returns:
        dict: crosses, intersection_length_m, intersection_ratio, crossed_uqa_codes
    """
    empty_result = {
        'crosses': False, 'intersection_length_m': 0, 
        'intersection_ratio': 0, 'crossed_uqa_codes': [],
    }
    
    if not path_latlon or len(path_latlon) < 2:
        return empty_result
    
    # polyline → shapely LineString (lon, lat 순서)
    line = LineString([(pt[1], pt[0]) for pt in path_latlon])
    
    if not line.intersects(avoid_union_param):
        return empty_result
    
    intersection = line.intersection(avoid_union_param)
    inter_length_m = intersection.length * 111000.0
    total_length_m = line.length * 111000.0
    ratio = inter_length_m / total_length_m if total_length_m > 0 else 0
    
    # 어느 UQA 코드와 교차?
    crossed_codes = []
    for code, union in avoid_union_by_code_param.items():
        if line.intersects(union):
            crossed_codes.append(code)
    
    return {
        'crosses': True,
        'intersection_length_m': inter_length_m,
        'intersection_ratio': ratio,
        'crossed_uqa_codes': crossed_codes,
    }


# === 동작 검증 — junggu·donggu에서 원본 K=3 vs 방법 C K=3 polyline 직접 비교 ===
print("[Tmap 보행자 호출 + 교차 검사 동작 검증]")
print("=" * 80)
print("방법 C가 K=3을 다르게 선택한 2개 OD에서 polyline 통과 효과 측정\n")

for district in ['junggu', 'donggu']:
    od = OD_CANDIDATES_V2[district]
    cands = od_buffer_results[district]['candidates']
    
    print(f"[{district}]")
    
    # 원본 K=3
    sel_orig, _ = algorithm_greedy_original(cands, od['O'], od['D'])
    wp_orig = [(row['latitude'], row['longitude']) 
               for _, row in sel_orig.iterrows()]
    
    # 방법 C K=3
    sel_v2, _ = algorithm_greedy_v2(cands, od['O'], od['D'],
                                      avoid_union, avoid_union_prepared)
    wp_v2 = [(row['latitude'], row['longitude'])
             for _, row in sel_v2.iterrows()]
    
    # 원본 polyline
    r_orig = get_pedestrian_route(od['O'], od['D'], waypoints=wp_orig)
    if r_orig['status'] == 'OK':
        c_orig = check_polyline_avoid_intersection(
            r_orig['path'], avoid_union, avoid_union_by_code)
        print(f"  원본 K=3 도보 polyline:")
        print(f"    거리 {r_orig['distance_m']/1000:.2f}km, 점 {len(r_orig['path'])}개")
        if c_orig['crosses']:
            print(f"    교차 {c_orig['intersection_length_m']:.0f}m "
                  f"({c_orig['intersection_ratio']*100:.2f}%), "
                  f"UQA {c_orig['crossed_uqa_codes']}")
        else:
            print(f"    교차 없음")
    else:
        print(f"  원본 polyline 호출 실패: {r_orig['status']}")
        continue
    
    # 방법 C polyline
    r_v2 = get_pedestrian_route(od['O'], od['D'], waypoints=wp_v2)
    if r_v2['status'] == 'OK':
        c_v2 = check_polyline_avoid_intersection(
            r_v2['path'], avoid_union, avoid_union_by_code)
        print(f"  방법 C K=3 도보 polyline:")
        print(f"    거리 {r_v2['distance_m']/1000:.2f}km, 점 {len(r_v2['path'])}개")
        if c_v2['crosses']:
            print(f"    교차 {c_v2['intersection_length_m']:.0f}m "
                  f"({c_v2['intersection_ratio']*100:.2f}%), "
                  f"UQA {c_v2['crossed_uqa_codes']}")
        else:
            print(f"    교차 없음")
        
        # 비교
        diff_m = c_v2['intersection_length_m'] - c_orig['intersection_length_m']
        if diff_m < 0:
            print(f"  방법 C 효과: 교차 {abs(diff_m):.0f}m 감소")
        elif diff_m > 0:
            print(f"  방법 C 효과: 교차 {diff_m:.0f}m 증가 (예상 외)")
        else:
            print(f"  방법 C 효과: 변화 없음")
    print()

[Tmap 보행자 호출 + 교차 검사 동작 검증]
방법 C가 K=3을 다르게 선택한 2개 OD에서 polyline 통과 효과 측정

[junggu]
  원본 K=3 도보 polyline:
    거리 4.87km, 점 244개
    교차 1973m (35.33%), UQA ['UQA320', 'UQA330']
  방법 C K=3 도보 polyline:
    거리 4.83km, 점 283개
    교차 1243m (22.08%), UQA ['UQA320', 'UQA330']
  방법 C 효과: 교차 730m 감소

[donggu]
  원본 K=3 도보 polyline:
    거리 7.40km, 점 492개
    교차 191m (2.26%), UQA ['UQA320']
  방법 C K=3 도보 polyline:
    거리 6.06km, 점 408개
    교차 191m (2.79%), UQA ['UQA320']
  방법 C 효과: 변화 없음



In [10]:
# 셀 10: swap 함수 — 방법 C K=3의 polyline이 통과 시 후보 교체

MAX_SWAP_ATTEMPTS = 5   # swap 최대 시도 횟수

def attempt_swap(scored_pool, initial_selection, origin, destination,
                  avoid_union_param, avoid_union_by_code_param,
                  max_attempts=MAX_SWAP_ATTEMPTS, verbose=False):
    """K=3 swap — polyline 통과 시 회피 페널티 가장 큰 핫스팟을 다음 후보로 교체.
    
    전략:
    1. 현재 K=3로 polyline 호출 + 교차 검사
    2. 교차 없으면 종료 (성공)
    3. 교차 시:
       a. 현재 K=3 중 avoid_penalty_i 가장 큰 후보 식별
       b. pool에서 K=3에 포함 안 된 다음 Score_i 순위 후보로 교체
       c. 새 K=3로 polyline 재호출
    4. max_attempts 초과 시 종료 (hard case)
    
    Args:
        scored_pool: compute_scores_with_avoidance 적용된 Top-15 풀
        initial_selection: 방법 C가 선정한 초기 K=3 DataFrame
        origin, destination: OD 좌표
        max_attempts: swap 시도 한도
    
    Returns:
        dict: final_selection, final_polyline, final_intersection, 
              final_distance_m, final_duration_ms, n_swaps, hard_case, swap_history
    """
    selected_ids = set(initial_selection['hotspot_id'])
    
    # 대체 후보: pool 중 초기 K=3에 안 들어간 후보를 Score_i 내림차순
    alternates = scored_pool[~scored_pool['hotspot_id'].isin(selected_ids)]
    alternates = alternates.sort_values('Score_i', ascending=False).reset_index(drop=True)
    
    current = initial_selection.copy().sort_values('Score_i', ascending=False).reset_index(drop=True)
    swap_history = []
    
    for attempt in range(max_attempts + 1):
        # 현재 K=3로 polyline 호출
        waypoints = [(row['latitude'], row['longitude']) 
                     for _, row in current.iterrows()]
        route = get_pedestrian_route(origin, destination, waypoints=waypoints)
        
        if route['status'] != 'OK':
            return {
                'final_selection': current, 'final_polyline': None,
                'final_intersection': None, 'final_distance_m': 0,
                'final_duration_ms': 0, 'n_swaps': attempt,
                'hard_case': True, 'swap_history': swap_history,
                'error': route['status'],
            }
        
        intersection = check_polyline_avoid_intersection(
            route['path'], avoid_union_param, avoid_union_by_code_param)
        
        # 교차 없으면 성공
        if not intersection['crosses']:
            if verbose:
                print(f"  swap {attempt}회 후 교차 해소")
            return {
                'final_selection': current,
                'final_polyline': route['path'],
                'final_intersection': intersection,
                'final_distance_m': route['distance_m'],
                'final_duration_ms': route['duration_ms'],
                'n_swaps': attempt,
                'hard_case': False,
                'swap_history': swap_history,
            }
        
        # 교차 발생 → swap 시도
        if attempt >= max_attempts or len(alternates) == 0:
            # 더 이상 swap 못 함 → hard case
            if verbose:
                print(f"  swap {attempt}회 후 hard case (한도 도달 또는 대체 후보 소진)")
            return {
                'final_selection': current,
                'final_polyline': route['path'],
                'final_intersection': intersection,
                'final_distance_m': route['distance_m'],
                'final_duration_ms': route['duration_ms'],
                'n_swaps': attempt,
                'hard_case': True,
                'swap_history': swap_history,
            }
        
        # 현재 K=3 중 페널티 가장 큰 후보 식별
        worst_idx = current['avoid_penalty_i'].idxmax()
        worst_id = current.loc[worst_idx, 'hotspot_id']
        worst_penalty = current.loc[worst_idx, 'avoid_penalty_i']
        
        # 다음 후보로 교체
        new_candidate = alternates.iloc[0]
        alternates = alternates.iloc[1:].reset_index(drop=True)
        
        if verbose:
            print(f"  swap #{attempt+1}: {worst_id} (penalty {worst_penalty:.2f}) "
                  f"→ {new_candidate['hotspot_id']}")
        
        swap_history.append({
            'attempt': attempt + 1,
            'removed': worst_id,
            'removed_penalty': worst_penalty,
            'added': new_candidate['hotspot_id'],
            'added_penalty': new_candidate['avoid_penalty_i'],
            'intersection_before_m': intersection['intersection_length_m'],
        })
        
        # K=3 갱신
        current = current.drop(index=worst_idx).reset_index(drop=True)
        new_row = new_candidate.to_frame().T
        current = pd.concat([current, new_row], ignore_index=True)
        current = current.sort_values('Score_i', ascending=False).reset_index(drop=True)
    
    # 도달 안 함
    return None


# === 동작 검증 — donggu에서 단독 swap 호출 ===
print("=" * 80)
print("[swap 함수 단독 동작 검증 — donggu]")
print("=" * 80)

district = 'donggu'
od = OD_CANDIDATES_V2[district]
cands = od_buffer_results[district]['candidates']

# 방법 C로 초기 K=3 선정
pool = cands.nlargest(min(BF_TOP_N, len(cands)), 'S_i')
scored_pool = compute_scores_with_avoidance(
    pool, od['O'], od['D'], avoid_union, avoid_union_prepared)
initial_sel = scored_pool.nlargest(K, 'Score_i')

print(f"  Buffer 후보: {len(cands)}곳 / S_i Top-{BF_TOP_N} 풀")
print(f"  초기 K=3 (방법 C 적용):")
for _, row in initial_sel.iterrows():
    print(f"    {row['hotspot_id']}: S_i={row['S_i']:.2f}, "
          f"D_i={row['D_i']:.2f}, penalty={row['avoid_penalty_i']:.2f}")

# swap 실행
print(f"\n  swap 시도 (verbose):")
swap_result = attempt_swap(
    scored_pool, initial_sel, od['O'], od['D'],
    avoid_union, avoid_union_by_code,
    verbose=True,
)

print(f"\n  최종 결과:")
print(f"    swap 횟수: {swap_result['n_swaps']}")
print(f"    hard case: {swap_result['hard_case']}")
if swap_result['final_intersection']:
    fi = swap_result['final_intersection']
    print(f"    최종 polyline 교차: {fi['intersection_length_m']:.0f}m "
          f"({fi['intersection_ratio']*100:.2f}%)")
print(f"    최종 도보 거리: {swap_result['final_distance_m']/1000:.2f}km")
print(f"    최종 K=3:")
for _, row in swap_result['final_selection'].iterrows():
    print(f"      {row['hotspot_id']}: S_i={row['S_i']:.2f}, "
          f"penalty={row['avoid_penalty_i']:.2f}")

[swap 함수 단독 동작 검증 — donggu]
  Buffer 후보: 46곳 / S_i Top-15 풀
  초기 K=3 (방법 C 적용):
    donggu_wydj1x1h_001: S_i=3.00, D_i=0.00, penalty=0.36
    donggu_wydj1wg6_001: S_i=2.50, D_i=0.00, penalty=0.41
    donggu_wydj1txm_001: S_i=2.50, D_i=0.01, penalty=0.45

  swap 시도 (verbose):
  swap #1: donggu_wydj1txm_001 (penalty 0.45) → donggu_wydj1x39_001
  swap #2: donggu_wydj1x39_001 (penalty 0.49) → donggu_wydj1vg9_001
  swap #3: donggu_wydj1vg9_001 (penalty 0.53) → donggu_wydj1ydk_001
  swap #4: donggu_wydj1ydk_001 (penalty 0.79) → donggu_wydj1x4r_001
  swap #5: donggu_wydj1x4r_001 (penalty 0.53) → donggu_wydj1txx_001
  swap 5회 후 hard case (한도 도달 또는 대체 후보 소진)

  최종 결과:
    swap 횟수: 5
    hard case: True
    최종 polyline 교차: 191m (2.82%)
    최종 도보 거리: 5.99km
    최종 K=3:
      donggu_wydj1x1h_001: S_i=3.00, penalty=0.36
      donggu_wydj1wg6_001: S_i=2.50, penalty=0.41
      donggu_wydj1txx_001: S_i=2.00, penalty=0.36


In [ ]:
# # 셀 11: 방법 C+swap 통합 알고리즘 함수 정의
# # visit_order는 Score_i 내림차순 고정 (evaluate_path 미사용)

# def algorithm_greedy_with_avoidance(candidates_df, origin, destination,
#                                       avoid_union_param, avoid_union_prepared_param,
#                                       avoid_union_by_code_param,
#                                       alpha=ALPHA, beta=BETA, gamma=GAMMA,
#                                       k=K, max_swap=MAX_SWAP_ATTEMPTS,
#                                       verbose=False):
#     """방법 C + swap 통합 알고리즘 (4번째 알고리즘 후보 E_v2).
    
#     파이프라인:
#     1. Buffer 후보 → S_i Top-15 풀
#     2. 방법 C: Score_i = α·S_i - β·D_i - γ·avoid_penalty_i
#     3. Score_i Top-K=3 초기 선정
#     4. K=3을 Score_i 내림차순으로 방문 (visit_order 최적화 없음)
#     5. Tmap polyline 호출 + 교차 검사
#     6. 교차 시 swap (penalty 큰 후보를 다음 Score 후보로 교체)
#     7. 최대 max_swap회까지 반복 후 hard case 종료
    
#     Args:
#         candidates_df: Buffer 후보 DataFrame
#         origin, destination: (lat, lon)
#         avoid_*: 셀 3에서 만든 회피 영역 객체들
#         gamma: 회피 페널티 가중치 (기본 1.0)
#         max_swap: swap 최대 시도 (기본 5)
#         verbose: 진행 출력 여부
    
#     Returns:
#         dict: final_selection, visit_order, final_polyline, final_intersection,
#               final_distance_m, final_duration_ms, n_swaps, hard_case,
#               swap_history, total_elapsed_ms
#     """
#     t0 = time.time()
    
#     # 1. S_i Top-15 풀
#     n = len(candidates_df)
#     if n == 0:
#         return None
#     pool = candidates_df.nlargest(min(BF_TOP_N, n), 'S_i')
    
#     # 2. 방법 C: Score_i 계산
#     scored_pool = compute_scores_with_avoidance(
#         pool, origin, destination,
#         avoid_union_param, avoid_union_prepared_param,
#         alpha=alpha, beta=beta, gamma=gamma,
#     )
    
#     # 3. Score_i Top-K=3 초기 선정
#     k_eff = min(k, n)
#     initial_selection = scored_pool.nlargest(k_eff, 'Score_i')
    
#     # 대체 후보: pool 중 K=3에 없는 것, Score_i 내림차순
#     alternates = scored_pool[~scored_pool['hotspot_id'].isin(initial_selection['hotspot_id'])]
#     alternates = alternates.sort_values('Score_i', ascending=False).reset_index(drop=True)
    
#     # 현재 K=3 (Score_i 내림차순으로 정렬)
#     current = initial_selection.copy().sort_values('Score_i', ascending=False).reset_index(drop=True)
#     swap_history = []
    
#     if verbose:
#         print(f"  초기 K=3 선정 완료. swap 루프 진입.")
    
#     # 4·5·6. polyline + swap 루프 (visit_order = Score_i 순서 고정)
#     for attempt in range(max_swap + 1):
#         # 4. K=3을 Score_i 내림차순(현재 정렬 상태)으로 방문
#         visit_order = current['hotspot_id'].tolist()
#         ordered_waypoints = [(row['latitude'], row['longitude'])
#                               for _, row in current.iterrows()]
        
#         # 5. Tmap polyline 호출
#         route = get_pedestrian_route(origin, destination, waypoints=ordered_waypoints)
#         if route['status'] != 'OK':
#             return {
#                 'final_selection': current, 'visit_order': visit_order,
#                 'final_polyline': None, 'final_intersection': None,
#                 'final_distance_m': 0, 'final_duration_ms': 0,
#                 'n_swaps': attempt, 'hard_case': True,
#                 'swap_history': swap_history,
#                 'total_elapsed_ms': (time.time() - t0) * 1000,
#                 'error': route['status'],
#             }
        
#         intersection = check_polyline_avoid_intersection(
#             route['path'], avoid_union_param, avoid_union_by_code_param)
        
#         if verbose:
#             cross_info = (f"{intersection['intersection_length_m']:.0f}m"
#                           if intersection['crosses'] else "없음")
#             print(f"  attempt {attempt}: 교차 {cross_info}")
        
#         # 6a. 교차 없으면 성공 종료
#         if not intersection['crosses']:
#             return {
#                 'final_selection': current,
#                 'visit_order': visit_order,
#                 'final_polyline': route['path'],
#                 'final_intersection': intersection,
#                 'final_distance_m': route['distance_m'],
#                 'final_duration_ms': route['duration_ms'],
#                 'n_swaps': attempt,
#                 'hard_case': False,
#                 'swap_history': swap_history,
#                 'total_elapsed_ms': (time.time() - t0) * 1000,
#             }
        
#         # 6b. swap 한도 도달 → hard case 종료
#         if attempt >= max_swap or len(alternates) == 0:
#             return {
#                 'final_selection': current,
#                 'visit_order': visit_order,
#                 'final_polyline': route['path'],
#                 'final_intersection': intersection,
#                 'final_distance_m': route['distance_m'],
#                 'final_duration_ms': route['duration_ms'],
#                 'n_swaps': attempt,
#                 'hard_case': True,
#                 'swap_history': swap_history,
#                 'total_elapsed_ms': (time.time() - t0) * 1000,
#             }
        
#         # 6c. swap 시도: penalty 가장 큰 후보 → 다음 Score 후보
#         worst_idx = current['avoid_penalty_i'].idxmax()
#         worst_id = current.loc[worst_idx, 'hotspot_id']
#         worst_penalty = current.loc[worst_idx, 'avoid_penalty_i']
        
#         new_candidate = alternates.iloc[0]
#         alternates = alternates.iloc[1:].reset_index(drop=True)
        
#         swap_history.append({
#             'attempt': attempt + 1,
#             'removed': worst_id,
#             'removed_penalty': worst_penalty,
#             'added': new_candidate['hotspot_id'],
#             'added_penalty': new_candidate['avoid_penalty_i'],
#             'intersection_before_m': intersection['intersection_length_m'],
#         })
        
#         # K=3 갱신 (Score_i 내림차순 재정렬)
#         current = current.drop(index=worst_idx).reset_index(drop=True)
#         new_row = new_candidate.to_frame().T
#         current = pd.concat([current, new_row], ignore_index=True)
#         current = current.sort_values('Score_i', ascending=False).reset_index(drop=True)
    
#     return None   # 도달 안 함


# print("[통합 알고리즘 함수 정의 완료]")
# print(f"  파이프라인: Buffer → S_i Top-{BF_TOP_N} → 방법 C Score_i Top-{K} → Tmap → swap")
# print(f"  visit_order: Score_i 내림차순 (최적화 없음)")
# print(f"  하이퍼파라미터: α={ALPHA}, β={BETA}, γ={GAMMA}, K={K}, max_swap={MAX_SWAP_ATTEMPTS}")


# # === 단독 동작 검증 — junggu ===
# print(f"\n{'=' * 80}")
# print(f"[통합 알고리즘 단독 검증 — junggu]")
# print(f"{'=' * 80}")

# district = 'junggu'
# od = OD_CANDIDATES_V2[district]
# cands = od_buffer_results[district]['candidates']

# result = algorithm_greedy_with_avoidance(
#     cands, od['O'], od['D'],
#     avoid_union, avoid_union_prepared, avoid_union_by_code,
#     verbose=True,
# )

# print(f"\n  실행 시간: {result['total_elapsed_ms']:.1f}ms")
# print(f"  swap 횟수: {result['n_swaps']}")
# print(f"  hard case: {result['hard_case']}")

# print(f"\n  최종 K=3 (방문 순서):")
# sel = result['final_selection']
# for i, hid in enumerate(result['visit_order'], 1):
#     row = sel[sel['hotspot_id'] == hid].iloc[0]
#     print(f"    {i}. {hid}: S_i={row['S_i']:.2f}, "
#           f"D_i={row['D_i']:.2f}, penalty={row['avoid_penalty_i']:.2f}, "
#           f"Score_i={row['Score_i']:.3f}")

# print(f"\n  최종 polyline:")
# print(f"    도보 거리: {result['final_distance_m']/1000:.2f}km")
# print(f"    도보 소요: {result['final_duration_ms']/60000:.1f}분")
# print(f"    점 수: {len(result['final_polyline'])}개")

# fi = result['final_intersection']
# if fi and fi['crosses']:
#     print(f"  회피 영역 교차: {fi['intersection_length_m']:.0f}m "
#           f"({fi['intersection_ratio']*100:.2f}%)")
#     print(f"  UQA 코드: {fi['crossed_uqa_codes']}")
# else:
#     print(f"  회피 영역 교차: 없음")

# if result['swap_history']:
#     print(f"\n  swap 이력:")
#     for h in result['swap_history']:
#         print(f"    #{h['attempt']}: {h['removed']} (p={h['removed_penalty']:.2f}) "
#               f"→ {h['added']}")

[통합 알고리즘 함수 정의 완료]
  파이프라인: Buffer → S_i Top-15 → 방법 C Score_i Top-3 → Tmap → swap
  visit_order: Score_i 내림차순 (최적화 없음)
  하이퍼파라미터: α=0.5, β=0.5, γ=1.0, K=3, max_swap=5

[통합 알고리즘 단독 검증 — junggu]
  초기 K=3 선정 완료. swap 루프 진입.
  attempt 0: 교차 1243m
  attempt 1: 교차 1243m
  attempt 2: 교차 1243m
  attempt 3: 교차 1243m
  attempt 4: 교차 1243m
  attempt 5: 교차 1243m

  실행 시간: 814.5ms
  swap 횟수: 5
  hard case: True

  최종 K=3 (방문 순서):
    1. junggu_wydj1qpm_001: S_i=3.50, D_i=1.35, penalty=0.27, Score_i=0.802
    2. junggu_wydj1w0t_001: S_i=3.00, D_i=1.72, penalty=0.27, Score_i=0.367
    3. junggu_wydj1qre_001: S_i=2.50, D_i=1.51, penalty=0.36, Score_i=0.134

  최종 polyline:
    도보 거리: 3.79km
    도보 소요: 50.9분
    점 수: 193개
  회피 영역 교차: 1243m (27.82%)
  UQA 코드: ['UQA320', 'UQA330']

  swap 이력:
    #1: junggu_wydj1mt3_001 (p=0.37) → junggu_wydj1mwu_001
    #2: junggu_wydj1mwu_001 (p=0.32) → junggu_wydj1mzk_001
    #3: junggu_wydj1mzk_001 (p=0.33) → junggu_wydj1mt0_001
    #4: junggu_wydj1mt0_001 (p=0.40) →

In [17]:
# 셀 11 (수정): 방법 C+swap 통합 알고리즘 + 최선 결과 추적 안전장치

def algorithm_greedy_with_avoidance(candidates_df, origin, destination,
                                      avoid_union_param, avoid_union_prepared_param,
                                      avoid_union_by_code_param,
                                      alpha=ALPHA, beta=BETA, gamma=GAMMA,
                                      k=K, max_swap=MAX_SWAP_ATTEMPTS,
                                      verbose=False):
    """방법 C + swap 통합 알고리즘 (4번째 알고리즘 후보 E_v2).
    
    파이프라인:
    1. Buffer 후보 → S_i Top-15 풀
    2. 방법 C: Score_i = α·S_i - β·D_i - γ·avoid_penalty_i
    3. Score_i Top-K=3 초기 선정
    4. K=3을 Score_i 내림차순으로 방문
    5. Tmap polyline 호출 + 교차 검사
    6. 교차 시 swap (penalty 큰 후보 → 다음 Score 후보)
    7. ★ 매 attempt마다 best 추적 → 최종적으로 best 반환
    8. 최대 max_swap회까지 반복 후 hard case 종료
    
    수정 (단계 6-α 발견 반영):
    - 최선 결과 추적 (best dict)
    - swap이 결과를 악화시켜도 baseline(또는 더 좋은) 결과 보장
    - 'best_from_attempt' 필드 추가 — 어느 attempt가 best였는지 명시
    
    Returns:
        dict: final_selection, visit_order, final_polyline, final_intersection,
              final_distance_m, final_duration_ms, n_swaps, hard_case,
              swap_history, total_elapsed_ms, best_from_attempt
    """
    t0 = time.time()
    
    # 1. S_i Top-15 풀
    n = len(candidates_df)
    if n == 0:
        return None
    pool = candidates_df.nlargest(min(BF_TOP_N, n), 'S_i')
    
    # 2. 방법 C: Score_i 계산
    scored_pool = compute_scores_with_avoidance(
        pool, origin, destination,
        avoid_union_param, avoid_union_prepared_param,
        alpha=alpha, beta=beta, gamma=gamma,
    )
    
    # 3. Score_i Top-K=3 초기 선정
    k_eff = min(k, n)
    initial_selection = scored_pool.nlargest(k_eff, 'Score_i')
    
    # 대체 후보: pool 중 K=3에 없는 것, Score_i 내림차순
    alternates = scored_pool[~scored_pool['hotspot_id'].isin(initial_selection['hotspot_id'])]
    alternates = alternates.sort_values('Score_i', ascending=False).reset_index(drop=True)
    
    current = initial_selection.copy().sort_values('Score_i', ascending=False).reset_index(drop=True)
    swap_history = []
    
    # ★ 최선 결과 추적 (NEW)
    best = {
        'cross_m': None,
        'selection': None,
        'visit_order': None,
        'polyline': None,
        'intersection': None,
        'distance_m': 0,
        'duration_ms': 0,
        'attempt_no': None,
    }
    
    if verbose:
        print(f"  초기 K=3 선정 완료. swap 루프 진입.")
    
    for attempt in range(max_swap + 1):
        # 4. K=3 방문 순서 = Score_i 내림차순
        visit_order = current['hotspot_id'].tolist()
        ordered_waypoints = [(row['latitude'], row['longitude'])
                              for _, row in current.iterrows()]
        
        # 5. Tmap polyline 호출
        route = get_pedestrian_route(origin, destination, waypoints=ordered_waypoints)
        if route['status'] != 'OK':
            # API 에러 → best가 있으면 best 반환
            if best['selection'] is not None:
                return {
                    'final_selection': best['selection'],
                    'visit_order': best['visit_order'],
                    'final_polyline': best['polyline'],
                    'final_intersection': best['intersection'],
                    'final_distance_m': best['distance_m'],
                    'final_duration_ms': best['duration_ms'],
                    'n_swaps': attempt,
                    'hard_case': True,
                    'swap_history': swap_history,
                    'total_elapsed_ms': (time.time() - t0) * 1000,
                    'best_from_attempt': best['attempt_no'],
                    'error': route['status'],
                }
            return {
                'final_selection': current, 'visit_order': visit_order,
                'final_polyline': None, 'final_intersection': None,
                'final_distance_m': 0, 'final_duration_ms': 0,
                'n_swaps': attempt, 'hard_case': True,
                'swap_history': swap_history,
                'total_elapsed_ms': (time.time() - t0) * 1000,
                'error': route['status'],
            }
        
        intersection = check_polyline_avoid_intersection(
            route['path'], avoid_union_param, avoid_union_by_code_param)
        current_cross_m = (intersection['intersection_length_m'] 
                           if intersection['crosses'] else 0)
        
        # ★ best 갱신 (NEW) — 교차 길이가 작을수록 좋음
        if best['cross_m'] is None or current_cross_m < best['cross_m']:
            best['cross_m'] = current_cross_m
            best['selection'] = current.copy()
            best['visit_order'] = visit_order
            best['polyline'] = route['path']
            best['intersection'] = intersection
            best['distance_m'] = route['distance_m']
            best['duration_ms'] = route['duration_ms']
            best['attempt_no'] = attempt
        
        if verbose:
            cross_info = (f"{intersection['intersection_length_m']:.0f}m"
                          if intersection['crosses'] else "없음")
            is_best = " ★best 갱신" if best['attempt_no'] == attempt else ""
            print(f"  attempt {attempt}: 교차 {cross_info}{is_best}")
        
        # 6a. 교차 없으면 성공 종료 (즉시 종료, best도 자동으로 현재)
        if not intersection['crosses']:
            return {
                'final_selection': current,
                'visit_order': visit_order,
                'final_polyline': route['path'],
                'final_intersection': intersection,
                'final_distance_m': route['distance_m'],
                'final_duration_ms': route['duration_ms'],
                'n_swaps': attempt,
                'hard_case': False,
                'swap_history': swap_history,
                'total_elapsed_ms': (time.time() - t0) * 1000,
                'best_from_attempt': attempt,
            }
        
        # 6b. swap 한도 도달 → ★ best 반환 (NEW)
        if attempt >= max_swap or len(alternates) == 0:
            return {
                'final_selection': best['selection'],
                'visit_order': best['visit_order'],
                'final_polyline': best['polyline'],
                'final_intersection': best['intersection'],
                'final_distance_m': best['distance_m'],
                'final_duration_ms': best['duration_ms'],
                'n_swaps': attempt,
                'hard_case': True,
                'swap_history': swap_history,
                'total_elapsed_ms': (time.time() - t0) * 1000,
                'best_from_attempt': best['attempt_no'],
            }
        
        # 6c. swap 시도
        worst_idx = current['avoid_penalty_i'].idxmax()
        worst_id = current.loc[worst_idx, 'hotspot_id']
        worst_penalty = current.loc[worst_idx, 'avoid_penalty_i']
        
        new_candidate = alternates.iloc[0]
        alternates = alternates.iloc[1:].reset_index(drop=True)
        
        swap_history.append({
            'attempt': attempt + 1,
            'removed': worst_id,
            'removed_penalty': worst_penalty,
            'added': new_candidate['hotspot_id'],
            'added_penalty': new_candidate['avoid_penalty_i'],
            'intersection_before_m': current_cross_m,
        })
        
        # K=3 갱신
        current = current.drop(index=worst_idx).reset_index(drop=True)
        new_row = new_candidate.to_frame().T
        current = pd.concat([current, new_row], ignore_index=True)
        current = current.sort_values('Score_i', ascending=False).reset_index(drop=True)
    
    return None   # 도달 안 함


print("[통합 알고리즘 함수 정의 완료 — 안전장치 추가]")
print(f"  파이프라인: Buffer → S_i Top-{BF_TOP_N} → 방법 C Score_i Top-{K} → Tmap → swap")
print(f"  ★ NEW: 최선 결과 추적. swap이 결과를 악화시킬 수 없음.")
print(f"  하이퍼파라미터: α={ALPHA}, β={BETA}, γ={GAMMA}, K={K}, max_swap={MAX_SWAP_ATTEMPTS}")


# === 단독 동작 검증 — junggu (best 추적 확인) ===
print(f"\n{'=' * 80}")
print(f"[통합 알고리즘 단독 검증 — junggu (best 추적 확인)]")
print(f"{'=' * 80}")

district = 'junggu'
od = OD_CANDIDATES_V2[district]
cands = od_buffer_results[district]['candidates']

result = algorithm_greedy_with_avoidance(
    cands, od['O'], od['D'],
    avoid_union, avoid_union_prepared, avoid_union_by_code,
    verbose=True,
)

print(f"\n  실행 시간: {result['total_elapsed_ms']:.1f}ms")
print(f"  swap 횟수: {result['n_swaps']}")
print(f"  hard case: {result['hard_case']}")
print(f"  ★ best 채택 시점: attempt {result['best_from_attempt']}")

print(f"\n  최종 K=3 (best 시점):")
sel = result['final_selection']
for i, hid in enumerate(result['visit_order'], 1):
    row = sel[sel['hotspot_id'] == hid].iloc[0]
    print(f"    {i}. {hid}: S_i={row['S_i']:.2f}, "
          f"D_i={row['D_i']:.2f}, penalty={row['avoid_penalty_i']:.2f}")

print(f"\n  최종 polyline:")
print(f"    도보 거리: {result['final_distance_m']/1000:.2f}km")
print(f"    회피 교차: {result['final_intersection']['intersection_length_m']:.0f}m"
      if result['final_intersection']['crosses'] else "    회피 교차: 없음")

[통합 알고리즘 함수 정의 완료 — 안전장치 추가]
  파이프라인: Buffer → S_i Top-15 → 방법 C Score_i Top-3 → Tmap → swap
  ★ NEW: 최선 결과 추적. swap이 결과를 악화시킬 수 없음.
  하이퍼파라미터: α=0.5, β=0.5, γ=1.0, K=3, max_swap=5

[통합 알고리즘 단독 검증 — junggu (best 추적 확인)]
  초기 K=3 선정 완료. swap 루프 진입.
  attempt 0: 교차 1243m ★best 갱신
  attempt 1: 교차 1243m
  attempt 2: 교차 1243m
  attempt 3: 교차 1243m
  attempt 4: 교차 1243m
  attempt 5: 교차 1243m

  실행 시간: 1067.5ms
  swap 횟수: 5
  hard case: True
  ★ best 채택 시점: attempt 0

  최종 K=3 (best 시점):
    1. junggu_wydj1qpm_001: S_i=3.50, D_i=1.35, penalty=0.27
    2. junggu_wydj1mt3_001: S_i=2.50, D_i=0.83, penalty=0.37
    3. junggu_wydj1w0t_001: S_i=3.00, D_i=1.72, penalty=0.27

  최종 polyline:
    도보 거리: 4.83km
    회피 교차: 1243m


In [19]:
# 셀 12: 9 OD 전체 적용 — 원본 vs 방법 C+swap 종합 비교
# 알고리즘 가치 정량 측정

print("=" * 90)
print("[9 OD 전체 비교 — 원본 vs 방법 C+swap]")
print("=" * 90)

comparison_data = []

for district, od in OD_CANDIDATES_V2.items():
    print(f"\n[{district}] OD: {od['O']} → {od['D']}")
    cands = od_buffer_results[district]['candidates']
    
    # ===== 원본 알고리즘 =====
    sel_orig, _ = algorithm_greedy_original(cands, od['O'], od['D'])
    wp_orig = [(row['latitude'], row['longitude'])
               for _, row in sel_orig.iterrows()]
    r_orig = get_pedestrian_route(od['O'], od['D'], waypoints=wp_orig)
    
    if r_orig['status'] == 'OK':
        inter_orig = check_polyline_avoid_intersection(
            r_orig['path'], avoid_union, avoid_union_by_code)
        orig_cross_m = inter_orig['intersection_length_m']
        orig_dist_km = r_orig['distance_m'] / 1000
        orig_ratio = inter_orig['intersection_ratio']
    else:
        orig_cross_m = 0
        orig_dist_km = 0
        orig_ratio = 0
        print(f"  원본 polyline 호출 실패: {r_orig['status']}")
    
    # ===== 방법 C+swap =====
    result_v2 = algorithm_greedy_with_avoidance(
        cands, od['O'], od['D'],
        avoid_union, avoid_union_prepared, avoid_union_by_code,
    )
    
    fi = result_v2['final_intersection']
    v2_cross_m = fi['intersection_length_m'] if fi else 0
    v2_dist_km = result_v2['final_distance_m'] / 1000
    v2_ratio = fi['intersection_ratio'] if fi else 0
    
    # ===== 비교 출력 =====
    diff_m = v2_cross_m - orig_cross_m
    if diff_m < -1:
        diff_label = f"↓ {abs(diff_m):.0f}m 감소"
    elif diff_m > 1:
        diff_label = f"↑ {diff_m:.0f}m 증가"
    else:
        diff_label = "= 동일"
    
    print(f"  원본:        교차 {orig_cross_m:>6.0f}m ({orig_ratio*100:>5.2f}%), "
          f"거리 {orig_dist_km:.2f}km")
    print(f"  방법 C+swap: 교차 {v2_cross_m:>6.0f}m ({v2_ratio*100:>5.2f}%), "
          f"거리 {v2_dist_km:.2f}km, "
          f"swap {result_v2['n_swaps']}회, "
          f"hard_case={result_v2['hard_case']}")
    print(f"  변화: {diff_label}")
    
    comparison_data.append({
        'district': district,
        'orig_cross_m': orig_cross_m,
        'orig_dist_km': orig_dist_km,
        'orig_ratio_pct': orig_ratio * 100,
        'v2_cross_m': v2_cross_m,
        'v2_dist_km': v2_dist_km,
        'v2_ratio_pct': v2_ratio * 100,
        'diff_m': diff_m,
        'n_swaps': result_v2['n_swaps'],
        'hard_case': result_v2['hard_case'],
        'elapsed_ms': result_v2['total_elapsed_ms'],
    })

# ===== 종합 통계 =====
df_compare = pd.DataFrame(comparison_data)

print(f"\n{'=' * 90}")
print(f"[종합 통계]")
print(f"{'=' * 90}")

# 1. 회피 통과 총합
total_orig = df_compare['orig_cross_m'].sum()
total_v2 = df_compare['v2_cross_m'].sum()
reduction = total_orig - total_v2
reduction_pct = (reduction / total_orig * 100) if total_orig > 0 else 0

print(f"\n[교차 길이 총합 — 회피 효과 측정]")
print(f"  원본:        {total_orig:>7.0f}m")
print(f"  방법 C+swap: {total_v2:>7.0f}m")
print(f"  감소량:      {reduction:>7.0f}m ({reduction_pct:+.1f}%)")

# 2. OD별 변화 분류
n_better = sum(1 for r in comparison_data if r['diff_m'] < -1)
n_worse = sum(1 for r in comparison_data if r['diff_m'] > 1)
n_same = 9 - n_better - n_worse

print(f"\n[OD별 변화 분포]")
print(f"  교차 감소 OD: {n_better}/9")
print(f"  교차 동일 OD: {n_same}/9")
print(f"  교차 증가 OD: {n_worse}/9 (예상 외)")

# 3. swap·hard case 통계
n_hard = int(df_compare['hard_case'].sum())
n_swap_triggered = int((df_compare['n_swaps'] > 0).sum())
total_swaps = int(df_compare['n_swaps'].sum())

print(f"\n[swap 통계]")
print(f"  swap 발생 OD: {n_swap_triggered}/9")
print(f"  hard case OD: {n_hard}/9")
print(f"  swap 총 횟수: {total_swaps}")
print(f"  평균 swap/OD: {total_swaps/9:.2f}")

# 4. Tmap 호출 총합
total_calls_orig = 9
total_calls_v2 = 9 + total_swaps

print(f"\n[Tmap 호출 비용]")
print(f"  원본 알고리즘 (단순):   {total_calls_orig}회")
print(f"  방법 C+swap:          {total_calls_v2}회")
print(f"  비용 증가율:           +{(total_calls_v2/total_calls_orig - 1)*100:.0f}%")
print(f"  평균 호출/OD:          {total_calls_v2/9:.2f}회")

# 5. 실행 시간
avg_elapsed = df_compare['elapsed_ms'].mean()
total_elapsed = df_compare['elapsed_ms'].sum()
print(f"\n[실행 시간]")
print(f"  평균/OD: {avg_elapsed:.1f}ms")
print(f"  총합:    {total_elapsed:.1f}ms ({total_elapsed/1000:.2f}초)")

# 6. 단계 5 결과와 비교
n_orig_crossing = sum(1 for r in comparison_data if r['orig_cross_m'] > 0)
n_v2_crossing = sum(1 for r in comparison_data if r['v2_cross_m'] > 0)

print(f"\n[단계 5 결과와 비교]")
print(f"  단계 5 Greedy 교차 발생: 2/9 (평균 602m)")
print(f"  본 셀 원본 교차 발생:     {n_orig_crossing}/9")
print(f"  본 셀 방법 C+swap 교차:   {n_v2_crossing}/9")

# 7. 상세 표
print(f"\n[OD별 상세 결과]")
print(df_compare[['district', 'orig_cross_m', 'v2_cross_m', 'diff_m', 
                   'n_swaps', 'hard_case']].to_string(index=False))

# CSV 저장
output_csv = OUTPUT_DIR / 'step6_baseline_vs_methodC_swap.csv'
df_compare.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f"\n[CSV 저장] {output_csv.resolve()}")
print(f"  크기: {output_csv.stat().st_size} bytes")

[9 OD 전체 비교 — 원본 vs 방법 C+swap]

[ganghwa] OD: (37.706505, 126.443398) → (37.732838, 126.517697)
  원본:        교차      0m ( 0.00%), 거리 14.78km
  방법 C+swap: 교차      0m ( 0.00%), 거리 14.78km, swap 0회, hard_case=False
  변화: = 동일

[junggu] OD: (37.473565, 126.621778) → (37.474748, 126.603258)
  원본:        교차   1973m (35.33%), 거리 4.87km
  방법 C+swap: 교차   1243m (22.08%), 거리 4.83km, swap 5회, hard_case=True
  변화: ↓ 730m 감소

[donggu] OD: (37.470733, 126.641914) → (37.48778, 126.621754)
  원본:        교차    191m ( 2.26%), 거리 7.40km
  방법 C+swap: 교차    191m ( 2.79%), 거리 6.06km, swap 5회, hard_case=True
  변화: = 동일

[michuhol] OD: (37.455015, 126.667435) → (37.435062, 126.690721)
  원본:        교차      0m ( 0.00%), 거리 5.09km
  방법 C+swap: 교차      0m ( 0.00%), 거리 5.09km, swap 0회, hard_case=False
  변화: = 동일

[seogu] OD: (37.545115, 126.676062) → (37.552748, 126.650239)
  원본:        교차      0m ( 0.00%), 거리 8.85km
  방법 C+swap: 교차      0m ( 0.00%), 거리 8.85km, swap 0회, hard_case=False
  변화: = 동일

[yeonsu] OD: (37.

### swap 관련 검증 : worst case OD 정의 + 검증

In [15]:
# 셀 13 (재재작성): 단일 폴리곤 기반 + 거리 자동 조정 worst case OD 생성

ENG_TO_KOR_DISTRICT = {v: k for k, v in KOR_TO_ENG_DISTRICT.items()}

def generate_od_for_single_polygon(district_eng, target_distance_km=3.0, uqa_filter=None):
    """군구 내 가장 큰 단일 회피 폴리곤을 짧은 방향 가로지름.
    
    Args:
        district_eng: 군구 영어명
        target_distance_km: 직선 거리 (km)
        uqa_filter: 특정 UQA 코드 리스트 (None이면 모든 UQA)
    
    Returns:
        ((O_lat, O_lon), (D_lat, D_lon), polygon_info) 또는 None
    """
    sigg_kor = ENG_TO_KOR_DISTRICT.get(district_eng)
    if not sigg_kor:
        return None
    
    polys = [p for p in avoid_polygons if sigg_kor in p['sigg_name']]
    if uqa_filter:
        polys = [p for p in polys if p['uqa_code'] in uqa_filter]
    if not polys:
        return None
    
    # 가장 큰 단일 폴리곤
    largest = max(polys, key=lambda p: p['geom'].area)
    geom = largest['geom']
    centroid = geom.centroid
    bounds = geom.bounds   # (lon_min, lat_min, lon_max, lat_max)
    
    width_deg = bounds[2] - bounds[0]
    height_deg = bounds[3] - bounds[1]
    half_dist_deg = (target_distance_km / 2) / 111.0
    
    # 짧은 방향으로 가로지름 (가로지르는 길이 최소화 → 도보 가능 거리)
    if width_deg < height_deg:
        # 동서 방향이 짧음 → 동서 방향으로 OD 배치
        O = (centroid.y, centroid.x - half_dist_deg)
        D = (centroid.y, centroid.x + half_dist_deg)
        cross_direction = 'EW'
    else:
        # 남북 방향이 짧음 → 남북 방향으로 OD 배치
        O = (centroid.y - half_dist_deg, centroid.x)
        D = (centroid.y + half_dist_deg, centroid.x)
        cross_direction = 'NS'
    
    return O, D, {
        'uqa_code': largest['uqa_code'],
        'uname': largest.get('uname'),
        'area_km2': geom.area * 12321,
        'cross_direction': cross_direction,
        'polygon_width_km': width_deg * 111 * math.cos(math.radians(37.5)),
        'polygon_height_km': height_deg * 111,
    }


def auto_generate_with_validation(district_eng, 
                                    distance_options=[3.0, 2.5, 3.5, 2.0, 4.0],
                                    uqa_filter=None):
    """거리 옵션을 순회하며 worst case 조건 만족하는 OD 자동 선정.
    
    검증 기준:
    1. OD 직선이 회피 영역과 교차
    2. 직선 거리 ≤ 6km
    3. Buffer 후보 ≥ 15곳
    """
    for target in distance_options:
        result = generate_od_for_single_polygon(district_eng, target, uqa_filter)
        if result is None:
            return None, "폴리곤 없음"
        O, D, polygon_info = result
        
        od_line = LineString([(O[1], O[0]), (D[1], D[0])])
        
        # 검증 1: 직선 교차
        if not od_line.intersects(avoid_union):
            continue   # 다음 거리 시도
        
        # 검증 2: 거리
        direct_km = haversine_km(O, D)
        if direct_km > 6.0:
            continue
        
        # 검증 3: Buffer 후보
        district_df = hotspots[hotspots['source_district'] == district_eng]
        buffer_r = get_buffer_radius(district_eng)
        cands = filter_candidates_by_ellipse_buffer(district_df, O, D, buffer_r)
        if len(cands) < 15:
            continue
        
        return (O, D, polygon_info, direct_km, cands, target), "성공"
    
    return None, "모든 거리 옵션 실패"


# === 5개 worst case OD 자동 생성 ===
print("=" * 90)
print("[자동 worst case OD 5개 생성 — 단일 폴리곤 기반]")
print("=" * 90)

WORST_CASE_SPECS = [
    ('W1_namdong_UQA310', 'namdong', ['UQA310'], '남동공단 UQA310 (전용공업)'),
    ('W2_seogu',          'seogu',    None,      '서구 가장 큰 폴리곤'),
    ('W3_bupyeong',       'bupyeong', None,      '부평구 가장 큰 폴리곤'),
    ('W4_junggu',         'junggu',   None,      '중구 가장 큰 폴리곤'),
    ('W5_michuhol',       'michuhol', None,      '미추홀구 가장 큰 폴리곤'),
]

WORST_CASE_OD = {}

for od_id, district, uqa_filter, desc in WORST_CASE_SPECS:
    result, status = auto_generate_with_validation(district, uqa_filter=uqa_filter)
    if result is None:
        print(f"\n  [{od_id}] {desc}: 자동 생성 실패 ({status})")
        continue
    
    O, D, info, direct_km, cands, used_distance = result
    WORST_CASE_OD[od_id] = {
        'O': O, 'D': D,
        'description': desc,
        'district_for_buffer': district,
        'polygon_info': info,
        'direct_km': direct_km,
        'used_distance_km': used_distance,
        'candidates': cands,
    }
    
    print(f"\n  [{od_id}] {desc}")
    print(f"    가장 큰 폴리곤: {info['uqa_code']} ({info['area_km2']:.2f}km², "
          f"동서 {info['polygon_width_km']:.2f}km × 남북 {info['polygon_height_km']:.2f}km)")
    print(f"    가로지름 방향: {info['cross_direction']} (짧은 쪽)")
    print(f"    O: ({O[0]:.4f}, {O[1]:.4f})")
    print(f"    D: ({D[0]:.4f}, {D[1]:.4f})")
    print(f"    직선 거리: {direct_km:.2f}km (시도된 거리: {used_distance}km)")


# === 자동 생성 OD 검증 ===
print(f"\n{'=' * 90}")
print(f"[자동 생성 OD 정밀 검증]")
print(f"{'=' * 90}")

worst_case_results = {}

for od_id, od in WORST_CASE_OD.items():
    od_line = LineString([(od['O'][1], od['O'][0]), (od['D'][1], od['D'][0])])
    inter = od_line.intersection(avoid_union)
    line_cross_m = inter.length * 111000 if not inter.is_empty else 0
    
    crossed_uqa = [code for code, union in avoid_union_by_code.items() 
                   if od_line.intersects(union)]
    
    n_avoid_in_buffer = int(od['candidates']['in_avoid_zone'].sum())
    
    print(f"\n[{od_id}] {od['description']}")
    print(f"  직선 거리: {od['direct_km']:.2f}km")
    print(f"  OD 직선 회피 영역 교차: {line_cross_m:.0f}m, UQA {crossed_uqa}")
    print(f"  Buffer 후보: {len(od['candidates'])}곳, 회피 영역 내부: {n_avoid_in_buffer}곳")
    
    # 판정
    if line_cross_m > 0 and od['direct_km'] <= 6.0 and len(od['candidates']) >= 15:
        verdict = "worst case 적합"
    else:
        verdict = "부적합"
    print(f"  판정: {verdict}")
    
    worst_case_results[od_id] = {
        'origin': od['O'],
        'destination': od['D'],
        'description': od['description'],
        'district_for_buffer': od['district_for_buffer'],
        'direct_km': od['direct_km'],
        'line_cross_m': line_cross_m,
        'crossed_uqa': crossed_uqa,
        'candidates': od['candidates'],
        'n_candidates': len(od['candidates']),
        'n_avoid_in_buffer': n_avoid_in_buffer,
        'verdict': verdict,
    }

# 종합
print(f"\n{'=' * 90}")
print(f"[검증 종합]")
print(f"{'=' * 90}")

n_appropriate = sum(1 for r in worst_case_results.values() if r['verdict'] == 'worst case 적합')
total_buffer_avoid = sum(r['n_avoid_in_buffer'] for r in worst_case_results.values())
total_line_cross = sum(r['line_cross_m'] for r in worst_case_results.values())

print(f"\n  worst case 적합: {n_appropriate}/{len(worst_case_results)}")
print(f"  Buffer 안 회피 핫스팟 총합: {total_buffer_avoid}곳")
print(f"  OD 직선 회피 영역 교차 총합: {total_line_cross:.0f}m")

inappropriate = [n for n, r in worst_case_results.items() if r['verdict'] != 'worst case 적합']
if inappropriate:
    print(f"\n  [재선정 필요] {len(inappropriate)}개:")
    for n in inappropriate:
        print(f"    {n}")

[자동 worst case OD 5개 생성 — 단일 폴리곤 기반]

  [W1_namdong_UQA310] 남동공단 UQA310 (전용공업)
    가장 큰 폴리곤: UQA310 (0.08km², 동서 0.32km × 남북 0.35km)
    가로지름 방향: NS (짧은 쪽)
    O: (37.3853, 126.7215)
    D: (37.4123, 126.7215)
    직선 거리: 3.01km (시도된 거리: 3.0km)

  [W2_seogu] 서구 가장 큰 폴리곤
    가장 큰 폴리곤: UQA320 (9.60km², 동서 3.80km × 남북 5.48km)
    가로지름 방향: EW (짧은 쪽)
    O: (37.4964, 126.6496)
    D: (37.4964, 126.6766)
    직선 거리: 2.38km (시도된 거리: 3.0km)

  [W3_bupyeong] 부평구 가장 큰 폴리곤
    가장 큰 폴리곤: UQA320 (2.17km², 동서 1.15km × 남북 1.77km)
    가로지름 방향: EW (짧은 쪽)
    O: (37.5161, 126.6976)
    D: (37.5161, 126.7336)
    직선 거리: 3.18km (시도된 거리: 4.0km)

  [W4_junggu] 중구 가장 큰 폴리곤
    가장 큰 폴리곤: UQA330 (8.41km², 동서 3.94km × 남북 5.75km)
    가로지름 방향: EW (짧은 쪽)
    O: (37.4543, 126.5961)
    D: (37.4543, 126.6321)
    직선 거리: 3.18km (시도된 거리: 4.0km)

  [W5_michuhol] 미추홀구 가장 큰 폴리곤
    가장 큰 폴리곤: UQA320 (1.43km², 동서 2.83km × 남북 1.59km)
    가로지름 방향: NS (짧은 쪽)
    O: (37.4626, 126.6692)
    D: (37.4896, 126.6692)
    직선 거리: 3.01k

In [18]:
# 셀 14: worst case 5 OD에 원본 vs 방법 C+swap 적용 — swap 가치 측정

print("=" * 90)
print("[worst case 5 OD — 원본 vs 방법 C+swap]")
print("=" * 90)

worst_case_comparison = []

for od_id, od_info in worst_case_results.items():
    if od_info['verdict'] != 'worst case 적합':
        continue
    
    print(f"\n[{od_id}] {od_info['description']}")
    
    origin = od_info['origin']
    destination = od_info['destination']
    cands = od_info['candidates']
    
    # ===== 원본 알고리즘 =====
    sel_orig, _ = algorithm_greedy_original(cands, origin, destination)
    wp_orig = [(row['latitude'], row['longitude']) for _, row in sel_orig.iterrows()]
    r_orig = get_pedestrian_route(origin, destination, waypoints=wp_orig)
    
    if r_orig['status'] != 'OK':
        print(f"  원본 polyline 호출 실패: {r_orig['status']}")
        continue
    
    inter_orig = check_polyline_avoid_intersection(
        r_orig['path'], avoid_union, avoid_union_by_code)
    orig_cross_m = inter_orig['intersection_length_m']
    orig_dist_km = r_orig['distance_m'] / 1000
    orig_ratio = inter_orig['intersection_ratio']
    
    # ===== 방법 C+swap =====
    result_v2 = algorithm_greedy_with_avoidance(
        cands, origin, destination,
        avoid_union, avoid_union_prepared, avoid_union_by_code,
    )
    
    fi = result_v2['final_intersection']
    v2_cross_m = fi['intersection_length_m'] if fi else 0
    v2_dist_km = result_v2['final_distance_m'] / 1000
    v2_ratio = fi['intersection_ratio'] if fi else 0
    
    # ===== 비교 출력 =====
    diff_m = v2_cross_m - orig_cross_m
    if diff_m < -1:
        diff_label = f"↓ {abs(diff_m):.0f}m 감소"
    elif diff_m > 1:
        diff_label = f"↑ {diff_m:.0f}m 증가"
    else:
        diff_label = "= 동일"
    
    print(f"  원본:        교차 {orig_cross_m:>6.0f}m ({orig_ratio*100:>5.2f}%), "
          f"거리 {orig_dist_km:.2f}km")
    print(f"  방법 C+swap: 교차 {v2_cross_m:>6.0f}m ({v2_ratio*100:>5.2f}%), "
          f"거리 {v2_dist_km:.2f}km, swap {result_v2['n_swaps']}회, "
          f"hard_case={result_v2['hard_case']}")
    print(f"  변화: {diff_label}")
    
    if result_v2['swap_history']:
        print(f"  swap 이력:")
        for h in result_v2['swap_history']:
            print(f"    #{h['attempt']}: {h['removed']} (p={h['removed_penalty']:.2f}) "
                  f"→ {h['added']}")
    
    worst_case_comparison.append({
        'od_id': od_id,
        'orig_cross_m': orig_cross_m,
        'orig_dist_km': orig_dist_km,
        'orig_ratio_pct': orig_ratio * 100,
        'v2_cross_m': v2_cross_m,
        'v2_dist_km': v2_dist_km,
        'v2_ratio_pct': v2_ratio * 100,
        'diff_m': diff_m,
        'n_swaps': result_v2['n_swaps'],
        'hard_case': result_v2['hard_case'],
        'elapsed_ms': result_v2['total_elapsed_ms'],
    })

# ===== 종합 통계 =====
df_wc = pd.DataFrame(worst_case_comparison)

print(f"\n{'=' * 90}")
print(f"[worst case 5 OD 종합 통계]")
print(f"{'=' * 90}")

total_orig = df_wc['orig_cross_m'].sum()
total_v2 = df_wc['v2_cross_m'].sum()
reduction = total_orig - total_v2
reduction_pct = (reduction / total_orig * 100) if total_orig > 0 else 0

n_better = sum(1 for r in worst_case_comparison if r['diff_m'] < -1)
n_same = sum(1 for r in worst_case_comparison if -1 <= r['diff_m'] <= 1)
n_worse = sum(1 for r in worst_case_comparison if r['diff_m'] > 1)

n_swap_triggered = int((df_wc['n_swaps'] > 0).sum())
n_hard = int(df_wc['hard_case'].sum())
total_swaps = int(df_wc['n_swaps'].sum())

print(f"\n[교차 길이 종합]")
print(f"  원본:        {total_orig:>7.0f}m")
print(f"  방법 C+swap: {total_v2:>7.0f}m")
print(f"  감소량:      {reduction:>7.0f}m ({reduction_pct:+.1f}%)")

print(f"\n[OD별 변화 분포]")
print(f"  교차 감소: {n_better}/5")
print(f"  교차 동일: {n_same}/5")
print(f"  교차 증가: {n_worse}/5")

print(f"\n[swap 통계]")
print(f"  swap 발생 OD:  {n_swap_triggered}/5")
print(f"  hard case OD:  {n_hard}/5")
print(f"  swap 총 횟수:  {total_swaps}")
print(f"  평균 swap/OD:  {total_swaps/5:.2f}")

print(f"\n[Tmap 호출 비용]")
total_calls_orig = 5
total_calls_v2 = 5 + total_swaps
print(f"  원본:        {total_calls_orig}회")
print(f"  방법 C+swap: {total_calls_v2}회 (+{(total_calls_v2/total_calls_orig-1)*100:.0f}%)")

print(f"\n[기존 9 OD 표본과 비교]")
print(f"  9 OD: 교차 감소 1/9 (junggu 730m), hard case 2/9, swap 트리거 2/9")
print(f"  worst case 5 OD: 교차 감소 {n_better}/5, hard case {n_hard}/5, "
      f"swap 트리거 {n_swap_triggered}/5")

# CSV 저장
output_csv = OUTPUT_DIR / 'step6_alpha_worst_case_vs_methodC_swap.csv'
df_wc.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f"\n[CSV 저장] {output_csv.resolve()}")

# 상세 표
print(f"\n[OD별 상세 결과]")
print(df_wc[['od_id', 'orig_cross_m', 'v2_cross_m', 'diff_m', 'n_swaps', 'hard_case']].to_string(index=False))

[worst case 5 OD — 원본 vs 방법 C+swap]

[W1_namdong_UQA310] 남동공단 UQA310 (전용공업)
  원본:        교차      0m ( 0.00%), 거리 4.74km
  방법 C+swap: 교차      0m ( 0.00%), 거리 4.78km, swap 0회, hard_case=False
  변화: = 동일

[W2_seogu] 서구 가장 큰 폴리곤
  원본:        교차   2710m (34.30%), 거리 7.03km
  방법 C+swap: 교차   2710m (34.30%), 거리 7.03km, swap 5회, hard_case=True
  변화: = 동일
  swap 이력:
    #1: seogu_wydj69fp_001 (p=2.39) → seogu_wydj6kqh_001
    #2: seogu_wydj63xz_001 (p=2.32) → seogu_wydj6kjx_001
    #3: seogu_wydj6knz_001 (p=2.22) → seogu_wydj6s1j_001
    #4: seogu_wydj6kjx_001 (p=2.22) → seogu_wydj6km6_001
    #5: seogu_wydj6km6_001 (p=2.19) → seogu_wydj6kmd_001

[W3_bupyeong] 부평구 가장 큰 폴리곤
  원본:        교차   4653m (32.14%), 거리 12.15km
  방법 C+swap: 교차    968m (12.74%), 거리 6.76km, swap 5회, hard_case=True
  변화: ↓ 3685m 감소
  swap 이력:
    #1: bupyeong_wydj7spk_001 (p=1.56) → bupyeong_wydj7u4d_001
    #2: bupyeong_wydj7u4d_001 (p=1.55) → bupyeong_wydj7heb_001
    #3: bupyeong_wydj7heb_001 (p=2.01) → bupyeong_wydj7he1_